<a href="https://colab.research.google.com/github/AnastasiyaPunko/Cygnus-Test-Pipeline/blob/main/BRCA_test2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
conda activate ngs

In [ ]:
conda list -n ngs

In [ ]:
fastqc BRCA_L000_R1.fq

In [ ]:
trimmomatic SE -phred33 BRCA_L000_R1.fq BRCA_L000_R1_trimm.fq.gz ILLUMINACLIP:/home/punko_a/genom/TruSeq3-SE.fa:2:30:10 LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36

In [ ]:
cutadapt -a AGATCGGAAGAGCACACGTCTGAACTCCAGTCAC -a "A{10}" -e 0.15 -m 36 -o BRCA_L000_R1_trimmed.fastq.gz BRCA_L000_R1_trimm.fq.gz

In [ ]:
fastqc BRCA_L000_R1_trimmed.fastq.gz

In [ ]:
bwa mem -R '@RG\tID:GSS5-0798\tSM:Hemato_pro_540_chip\tPL:ILLUMINA' -t 3 /home/punko_a/genom/Homo_sapiens_assembly37.fasta BRCA_L000_R1_trimmed.fastq.gz -o BRCA.sam

In [ ]:
samtools view -Sb BRCA.sam > BRCA.bam

In [ ]:
samtools sort -m 3G -o BRCA_sort.bam BRCA.bam

In [ ]:
samtools index BRCA_sort.bam

In [ ]:
#In high-depth targeted regions like BRCA, SE deduplication based on coordinates often leads to significant "over-killing" of real biological reads, which reduces sensitivity for low-frequency variants.
#gatk MarkDuplicates --INPUT /home/punko_a/BRCA_test/BRCA_sort.bam --TMP_DIR tmp --ASSUME_SORT_ORDER coordinate --CREATE_INDEX true --METRICS_FILE file --OUTPUT /home/punko_a/BRCA_test/BRCA_sort_dupl.bam

# Somatic

In [ ]:
gatk Mutect2 -R /home/punko_a/genom/Homo_sapiens_assembly37.fasta -I BRCA_sort.bam --disable-read-filter MateOnSameContigOrNoMappedMateReadFilter -O BRCA_sort_somatic.vcf.gz

In [ ]:
gatk FilterMutectCalls -R /home/punko_a/genom/Homo_sapiens_assembly37.fasta -V BRCA_sort_somatic.vcf.gz -O BRCA_sort_somatic_filtered.vcf.gz

In [ ]:
#nano BRCA_sort_somatic_filtered.vcf.gz.filteringStats.tsv

In [ ]:
#bcftools view BRCA_sort_somatic_filtered.vcf.gz -h

In [ ]:
#VarScan2
samtools mpileup -f /home/punko_a/genom/Homo_sapiens_assembly37.fasta BRCA_sort.bam > BRCA.mpileup

varscan mpileup2cns BRCA.mpileup --min-coverage 20 --min-var-freq 0.01 --p-value 0.05 --output-vcf 1 > BRCA_somatic_varscan.vcf

# Germline

In [ ]:
gatk --java-options "-Xmx4g" HaplotypeCaller -R /home/punko_a/genom/Homo_sapiens_assembly37.fasta -I BRCA_sort.bam -O BRCA_sort_germline.vcf.gz -bamout BRCA_sort_var.bam

In [ ]:
#VarScan2
samtools mpileup -f /home/punko_a/genom/Homo_sapiens_assembly37.fasta -q 20 -Q 20 -B BRCA_sort.bam > BRCA_germline.mpileup

#SNP calling
varscan mpileup2snp BRCA_germline.mpileup --min-coverage 20 --min-reads2 4 --min-var-freq 0.20 --p-value 0.05 --output-vcf 1 > BRCA_varscan_snps.vcf

#InDel calling
varscan mpileup2indel BRCA_germline.mpileup --min-coverage 20 --min-reads2 4 --min-var-freq 0.20 --p-value 0.05 --output-vcf 1 > BRCA_varscan_indels.vcf

In [ ]:
#Merge SNP+InDel
#bcftools concat -a -Ov -o BRCA_varscan_all.vcf BRCA_varscan_snps.vcf BRCA_varscan_indels.vcf
gatk MergeVcfs -I BRCA_varscan_snps.vcf -I BRCA_varscan_indels.vcf -O BRCA_varscan_all.vcf -D /home/punko_a/genom/Homo_sapiens_assembly37.dict

In [ ]:
#LoFreq
lofreq indelqual --dindel -f /home/punko_a/genom/Homo_sapiens_assembly37.fasta -o BRCA_indelq.bam BRCA_sort.bam

samtools index BRCA_indelq.bam

lofreq call-parallel --pp-threads 3 -f /home/punko_a/genom/Homo_sapiens_assembly37.fasta -o BRCA_lofreq.vcf BRCA_indelq.bam

lofreq filter -i BRCA_lofreq.vcf -o BRCA_lofreq_filtered.vcf --cov-min 20

# Annotation

In [ ]:
perl /home/punko_a/genom/annovar/table_annovar.pl BRCA_sort_somatic_filtered.vcf.gz /home/punko_a/genom/annovar/humandb/ -buildver hg19 -out BRCA_somatic -protocol refGeneWithVer -operation g -remove -polish -vcfinput -nastring .